# Exercise 1 — Skip-gram word2vec on the IMDB dataset

The Shakespeare corpus is replaced by the 50,000 IMDB movie reviews in `IMDB_Dataset.csv`.

Pipeline:
1. Clean the reviews and split them into sentences.
2. Vectorize sentences with `TextVectorization`.
3. Generate skip-gram pairs (with frequent-word subsampling) and negative samples.
4. Train a skip-gram negative-sampling model (target embedding · context embedding).
5. Export `vectors.tsv` + `metadata.tsv` and load them into https://projector.tensorflow.org/.

In [1]:
import io
import os
import re
import string
import time

import numpy as np
import pandas as pd
import tqdm

import tensorflow as tf
from tensorflow.keras import layers

SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)
print(tf.__version__)

2.21.0


## 1. Load and clean the IMDB corpus

Reviews contain HTML line breaks (`<br />`). Each review is split into sentences so that a skip-gram
window never spans two unrelated sentences, which mirrors the line-per-sample setup of the lab.

In [2]:
df = pd.read_csv('IMDB_Dataset.csv')
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
def split_sentences(review):
  review = re.sub(r'<br\s*/?>', ' ', review)
  return [s.strip() for s in re.split(r'[.!?]+', review) if len(s.split()) >= 3]

sentences = [s for review in df['review'] for s in split_sentences(review)]
print(f'{len(sentences):,} sentences')
for s in sentences[:5]:
  print('-', s)

628,206 sentences
- One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked
- They are right, as this is exactly what happened with me
- The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO
- Trust me, this is not a show for the faint hearted or timid
- This show pulls no punches with regards to drugs, sex or violence


## 2. Vectorize sentences

Same `custom_standardization` as the lab (lowercase + strip punctuation). The vocabulary is larger
(10,000) because IMDB is far bigger than the Shakespeare file; sequences are capped at 20 tokens.

In [4]:
def custom_standardization(input_data):
  lowercase = tf.strings.lower(input_data)
  return tf.strings.regex_replace(lowercase, '[%s]' % re.escape(string.punctuation), '')

vocab_size = 10000
sequence_length = 20

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length)

text_ds = tf.data.Dataset.from_tensor_slices(sentences)
vectorize_layer.adapt(text_ds.batch(1024))

inverse_vocab = vectorize_layer.get_vocabulary()
print(inverse_vocab[:20])

['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('in'), np.str_('it'), np.str_('i'), np.str_('this'), np.str_('that'), np.str_('was'), np.str_('as'), np.str_('with'), np.str_('for'), np.str_('movie'), np.str_('but'), np.str_('film')]


In [5]:
text_vector_ds = text_ds.batch(1024).prefetch(AUTOTUNE).map(vectorize_layer).unbatch()
sequences = list(text_vector_ds.as_numpy_iterator())
print(len(sequences))
for seq in sequences[:3]:
  print(f'{seq} => {[inverse_vocab[i] for i in seq]}')

628206
[  28    5    2   76 1954   45 1061   12  101  144   40  446 3101  389
  464   26 3138    0    0    0] => [np.str_('one'), np.str_('of'), np.str_('the'), np.str_('other'), np.str_('reviewers'), np.str_('has'), np.str_('mentioned'), np.str_('that'), np.str_('after'), np.str_('watching'), np.str_('just'), np.str_('1'), np.str_('oz'), np.str_('episode'), np.str_('youll'), np.str_('be'), np.str_('hooked'), '', '', '']
[ 34  23 209  14  11   7 606  48 586  15  69   0   0   0   0   0   0   0
   0   0] => [np.str_('they'), np.str_('are'), np.str_('right'), np.str_('as'), np.str_('this'), np.str_('is'), np.str_('exactly'), np.str_('what'), np.str_('happened'), np.str_('with'), np.str_('me'), '', '', '', '', '', '', '', '', '']
[   2   88  145   12 3262   69   42 3101   13   29 5393    3    1  133
    5  574   61  283    8  209] => [np.str_('the'), np.str_('first'), np.str_('thing'), np.str_('that'), np.str_('struck'), np.str_('me'), np.str_('about'), np.str_('oz'), np.str_('was'), np.st

## 3. Generate training data

Positive pairs come from `tf.keras.preprocessing.sequence.skipgrams` with the Zipf-based sampling
table, exactly as in the lab.

The lab calls `tf.random.log_uniform_candidate_sampler` once per positive pair. That is fine for
Shakespeare but would take hours for the millions of IMDB pairs, so negatives are drawn here in one
vectorized NumPy call from the **same log-uniform (Zipf) distribution**:
`P(k) = (log(k + 2) - log(k + 1)) / log(range_max + 1)`.
Negatives that collide with the true context word are resampled.

In [6]:
def log_uniform_probs(range_max):
  k = np.arange(range_max)
  return (np.log(k + 2) - np.log(k + 1)) / np.log(range_max + 1)

def generate_training_data(sequences, window_size, num_ns, vocab_size, seed):
  sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(vocab_size)
  targets, pos_contexts = [], []

  for sequence in tqdm.tqdm(sequences, mininterval=60):
    positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
        sequence,
        vocabulary_size=vocab_size,
        sampling_table=sampling_table,
        window_size=window_size,
        negative_samples=0,
        seed=seed)
    for target_word, context_word in positive_skip_grams:
      targets.append(target_word)
      pos_contexts.append(context_word)

  targets = np.array(targets, dtype=np.int32)
  pos_contexts = np.array(pos_contexts, dtype=np.int32)

  # Vectorized negative sampling from the log-uniform distribution.
  probs = log_uniform_probs(vocab_size)
  local_rng = np.random.default_rng(seed)
  negatives = local_rng.choice(vocab_size, size=(len(targets), num_ns), p=probs).astype(np.int32)
  clash = negatives == pos_contexts[:, None]
  while clash.any():
    negatives[clash] = local_rng.choice(vocab_size, size=clash.sum(), p=probs)
    clash = negatives == pos_contexts[:, None]

  contexts = np.concatenate([pos_contexts[:, None], negatives], axis=1)
  labels = np.zeros_like(contexts)
  labels[:, 0] = 1  # first context is the true one, the rest are negatives
  return targets, contexts, labels

In [7]:
start = time.time()
targets, contexts, labels = generate_training_data(
    sequences=sequences,
    window_size=2,
    num_ns=4,
    vocab_size=vocab_size,
    seed=SEED)
print(f'\n{time.time() - start:.0f}s')
print(f'targets.shape: {targets.shape}')
print(f'contexts.shape: {contexts.shape}')
print(f'labels.shape: {labels.shape}')


  0%|          | 0/628206 [00:00<?, ?it/s]


100%|██████████| 628206/628206 [00:12<00:00, 50302.58it/s]


15s
targets.shape: (5547029,)
contexts.shape: (5547029, 5)
labels.shape: (5547029, 5)


In [8]:
i = 0
print('target  :', inverse_vocab[targets[i]])
print('contexts:', [inverse_vocab[c] for c in contexts[i]])
print('labels  :', labels[i])

target  : 1
contexts: [np.str_('episode'), np.str_('beautifully'), np.str_('up'), np.str_('entertain'), np.str_('jokes')]
labels  : [1 0 0 0 0]


In [9]:
BATCH_SIZE = 1024
BUFFER_SIZE = 100000
dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), labels))
dataset = dataset.shuffle(BUFFER_SIZE, seed=SEED).batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.cache().prefetch(buffer_size=AUTOTUNE)
print(dataset)

<_PrefetchDataset element_spec=((TensorSpec(shape=(1024,), dtype=tf.int32, name=None), TensorSpec(shape=(1024, 5), dtype=tf.int32, name=None)), TensorSpec(shape=(1024, 5), dtype=tf.int32, name=None))>


## 4. Model and training

Same subclassed model as the lab: a `target_embedding` (the word vectors we keep) and a
`context_embedding`; the logit for each (target, context) pair is their dot product.

In [10]:
class Word2Vec(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.target_embedding = layers.Embedding(vocab_size, embedding_dim, name='w2v_embedding')
    self.context_embedding = layers.Embedding(vocab_size, embedding_dim)

  def call(self, pair):
    target, context = pair
    if len(target.shape) == 2:
      target = tf.squeeze(target, axis=1)
    word_emb = self.target_embedding(target)        # (batch, embed)
    context_emb = self.context_embedding(context)   # (batch, context, embed)
    return tf.einsum('be,bce->bc', word_emb, context_emb)  # (batch, context)

In [11]:
embedding_dim = 128
EPOCHS = 10
word2vec = Word2Vec(vocab_size, embedding_dim)
word2vec.compile(optimizer='adam',
                 loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
history = word2vec.fit(dataset, epochs=EPOCHS, verbose=2)

Epoch 1/10


/Users/kb/Documents/study/master/nlp/nlp/exercies/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


5417/5417 - 21s - 4ms/step - accuracy: 0.4527 - loss: 1.3122


Epoch 2/10


5417/5417 - 20s - 4ms/step - accuracy: 0.5297 - loss: 1.1625


Epoch 3/10


... (11 outputs left out)

Epoch 9/10


5417/5417 - 20s - 4ms/step - accuracy: 0.6294 - loss: 0.9284


Epoch 10/10


5417/5417 - 20s - 4ms/step - accuracy: 0.6348 - loss: 0.9151


## 5. Sanity check — nearest neighbours

Before exporting, check that cosine neighbours make sense for a few movie-review words.

In [12]:
weights = word2vec.get_layer('w2v_embedding').get_weights()[0]
vocab = vectorize_layer.get_vocabulary()
word_to_id = {w: i for i, w in enumerate(vocab)}
normed = weights / np.linalg.norm(weights, axis=1, keepdims=True)

def nearest(word, k=8):
  sims = normed @ normed[word_to_id[word]]
  sims[:2] = -1  # skip padding and [UNK]
  best = np.argsort(-sims)[1:k + 1]
  return [(vocab[i], round(float(sims[i]), 2)) for i in best]

for w in ['good', 'terrible', 'funny', 'actor', 'horror', 'music', 'boring', 'scary']:
  print(f'{w:>9}:', ', '.join(f'{n} {s}' for n, s in nearest(w)))

     good: great 0.61, decent 0.59, bad 0.59, nice 0.58, neat 0.5, fine 0.49, solid 0.47, pleasant 0.45
 terrible: horrible 0.65, horrendous 0.54, awful 0.52, bad 0.51, fantastic 0.5, lousy 0.47, atrocious 0.47, awesome 0.47
    funny: amusing 0.58, scary 0.56, entertaining 0.55, hilarious 0.53, watchable 0.5, disturbing 0.5, interesting 0.48, enjoyable 0.48
    actor: actress 0.66, comedian 0.54, actors 0.46, performer 0.45, villain 0.44, player 0.43, dancer 0.42, actresses 0.41
   horror: scifi 0.59, slasher 0.53, madefortv 0.5, disaster 0.47, campy 0.47, indie 0.46, lowbudget 0.46, blaxploitation 0.45
    music: cinematography 0.57, photography 0.57, soundtrack 0.52, acting 0.51, editing 0.51, lighting 0.5, animation 0.49, artwork 0.49
   boring: dull 0.59, predictable 0.53, preachy 0.5, unoriginal 0.49, derivative 0.48, confusing 0.48, corny 0.48, laughable 0.48
    scary: funny 0.56, cute 0.48, shocking 0.47, gory 0.46, suspenseful 0.46, exciting 0.46, disturbing 0.45, frightening

## 6. Export for the Embedding Projector

Index 0 (padding) and 1 (`[UNK]`) are skipped.

In [13]:
os.makedirs('output', exist_ok=True)
with io.open('output/vectors.tsv', 'w', encoding='utf-8') as out_v, \
     io.open('output/metadata.tsv', 'w', encoding='utf-8') as out_m:
  for index, word in enumerate(vocab):
    if index < 2:
      continue
    out_v.write('\t'.join(f'{x:.5f}' for x in weights[index]) + '\n')
    out_m.write(word + '\n')
print(len(vocab) - 2, 'words exported to output/vectors.tsv and output/metadata.tsv')

9998 words exported to output/vectors.tsv and output/metadata.tsv


Open https://projector.tensorflow.org/ → **Load** → upload `output/vectors.tsv` (step 1) and
`output/metadata.tsv` (step 2). Then pick PCA / t-SNE / UMAP and search for words such as
`good`, `horror` or `actor` to see their neighbourhoods.